# rl_train.py

In [ ]:
# rl_train.py
# Обучение RL-агентов на синтетических матрицах ATSP для n=12 и n=29.

import torch
import numpy as np
import os
import json
import shutil
import time
import matplotlib.pyplot as plt
from tqdm import tqdm

from rl_environment import ATSPEnvironment
from rl_agents import (
    PolicyGradientAgent, A2CAgent, PPOAgent, DQNAgent, DoubleDQNAgent, SACAgent,
    EnhancedPolicyGradientAgent, EnhancedA2CAgent, EnhancedPPOAgent,
    ImprovedDQNAgent, ImprovedDoubleDQNAgent, EnhancedSACAgent
)
from utils import compute_rl_training_metrics

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def generate_atsp_data(batch_size, n_nodes, asym_ratio=0.3, device=DEVICE):
    """Генерирует синтетические асимметричные матрицы расстояний."""
    dist_matrix = torch.rand(batch_size, n_nodes, n_nodes, device=device)
    asym_mask = torch.rand(batch_size, n_nodes, n_nodes, device=device) < asym_ratio
    dist_matrix[asym_mask] = torch.rand(asym_mask.sum(), device=device)
    sym_mask = ~asym_mask
    dist_matrix[sym_mask] = (dist_matrix[sym_mask] + dist_matrix.transpose(1, 2)[sym_mask]) / 2
    return dist_matrix

In [ ]:
def train_rl_agent(agent, n_nodes, num_episodes, save_dir, agent_name, log_freq=100):
    """
    Обучает одного RL-агента в течение num_episodes эпизодов.
    Возвращает список длин маршрутов по эпизодам и лучшую дистанцию.
    """
    os.makedirs(save_dir, exist_ok=True)
    start_time = time.time()

    best_distance = float('inf')
    episode_distances = []
    episode_rewards = []

    pbar = tqdm(range(num_episodes), desc=agent_name)
    for episode in pbar:
        # Генерируем одну матрицу ATSP
        dist_matrix = generate_atsp_data(1, n_nodes).squeeze(0).cpu().numpy()
        env = ATSPEnvironment(dist_matrix)
        state = env.reset()
        done = False
        episode_reward = 0.0

        while not done:
            action = agent.select_action(state)
            next_state, reward, done, _ = env.step(action)

            # Сохраняем переход для replay-based агентов (DQN, SAC и их улучшенные версии)
            if hasattr(agent, 'store_experience'):
                agent.store_experience(state, action, reward, next_state, done)

            # Для Policy Gradient и A2C накапливаем награды в списке агента
            if hasattr(agent, 'rewards'):
                agent.rewards.append(reward)

            # Для PPO-агентов передаём награду последнего шага через специальный метод
            if isinstance(agent, (PPOAgent, EnhancedPPOAgent)):
                agent.set_last_reward(reward, done)

            state = next_state
            episode_reward += reward

        # Обновление политики (один вызов после завершения эпизода)
        # PPO-агенты имеют всю информацию внутри memory и обновятся без аргументов
        if hasattr(agent, 'update'):
            agent.update()   # для PPO теперь вызывается без параметров

        dist_km = env.get_route_info()['distance_km']
        episode_distances.append(dist_km)
        episode_rewards.append(episode_reward)

        # Сохраняем лучшую модель
        if dist_km < best_distance:
            best_distance = dist_km
            torch.save(agent, os.path.join(save_dir, f"{agent_name}_best.pt"))

        # Логирование прогресса
        if (episode + 1) % log_freq == 0:
            avg_last_100 = np.mean(episode_distances[-100:]) if len(episode_distances) >= 100 else np.mean(episode_distances)
            pbar.set_postfix({
                'best': f'{best_distance:.2f}',
                'avg100': f'{avg_last_100:.2f}'
            })

    training_time = time.time() - start_time
    torch.save(agent, os.path.join(save_dir, f"{agent_name}_final.pt"))

    # Расширенные метрики обучения
    metrics = compute_rl_training_metrics(episode_distances, window=100)
    metrics['training_time_sec'] = training_time
    metrics['episode_rewards'] = [float(x) for x in episode_rewards]

    metrics_path = os.path.join(save_dir, f"{agent_name}_metrics.json")
    with open(metrics_path, 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    print(f"Метрики сохранены: {metrics_path}")

    # График кривой обучения
    plt.figure(figsize=(10, 5))
    plt.plot(episode_distances, alpha=0.5, label='Длина маршрута')
    window = 100
    if len(episode_distances) >= window:
        ma = np.convolve(episode_distances, np.ones(window)/window, mode='valid')
        plt.plot(range(window-1, len(episode_distances)), ma, label=f'Скользящее среднее ({window})')
    plt.xlabel('Эпизод')
    plt.ylabel('Длина маршрута (км)')
    plt.title(f'Кривая обучения: {agent_name} (n={n_nodes})')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plot_path = os.path.join(save_dir, f"{agent_name}_learning_curve.png")
    plt.savefig(plot_path, dpi=150)
    plt.close()

    return episode_distances, best_distance

In [ ]:
# Реестр всех доступных агентов
AGENT_REGISTRY = {
    'PolicyGradient': PolicyGradientAgent,
    'A2C': A2CAgent,
    'PPO': PPOAgent,
    'DQN': DQNAgent,
    'DoubleDQN': DoubleDQNAgent,
    'SAC': SACAgent,
    'EnhancedPolicyGradient': EnhancedPolicyGradientAgent,
    'EnhancedA2C': EnhancedA2CAgent,
    'EnhancedPPO': EnhancedPPOAgent,
    'ImprovedDQN': ImprovedDQNAgent,
    'ImprovedDoubleDQN': ImprovedDoubleDQNAgent,
    'EnhancedSAC': EnhancedSACAgent,
}

In [ ]:
if __name__ == "__main__":
    NODE_SIZES = [12, 29]
    AGENT_NAMES = [
        # Базовые
        'PolicyGradient', 'A2C', 'PPO', 'DQN', 'DoubleDQN', 'SAC',
        # Улучшенные
        'EnhancedPolicyGradient', 'EnhancedA2C', 'EnhancedPPO',
        'ImprovedDQN', 'ImprovedDoubleDQN', 'EnhancedSAC',
    ]
    NUM_EPISODES = 1500

    for n_nodes in NODE_SIZES:
        print(f"\nTraining RL agents for ATSP with {n_nodes} nodes")

        base_save_dir = f"models_rl_{n_nodes}"
        os.makedirs(base_save_dir, exist_ok=True)

        for agent_name in AGENT_NAMES:
            if agent_name not in AGENT_REGISTRY:
                print(f"Unknown agent: {agent_name}, skipping")
                continue

            print(f"\n  Training {agent_name} on {n_nodes} nodes")
            agent_class = AGENT_REGISTRY[agent_name]
            agent = agent_class(n_nodes)

            agent_save_dir = os.path.join(base_save_dir, agent_name)
            train_rl_agent(
                agent=agent,
                n_nodes=n_nodes,
                num_episodes=NUM_EPISODES,
                save_dir=agent_save_dir,
                agent_name=agent_name
            )

        # Подготовка zip-архивов: отдельно базовые, отдельно улучшенные
        basic_agents = ['PolicyGradient', 'A2C', 'PPO', 'DQN', 'DoubleDQN', 'SAC']
        enhanced_agents = ['EnhancedPolicyGradient', 'EnhancedA2C', 'EnhancedPPO',
                           'ImprovedDQN', 'ImprovedDoubleDQN', 'EnhancedSAC']

        basic_zip_dir = f"models_rl_{n_nodes}_basic"
        enhanced_zip_dir = f"models_rl_{n_nodes}_enhanced"
        os.makedirs(basic_zip_dir, exist_ok=True)
        os.makedirs(enhanced_zip_dir, exist_ok=True)

        for ag in basic_agents:
            src = os.path.join(base_save_dir, ag)
            if os.path.exists(src):
                shutil.copytree(src, os.path.join(basic_zip_dir, ag), dirs_exist_ok=True)
        for ag in enhanced_agents:
            src = os.path.join(base_save_dir, ag)
            if os.path.exists(src):
                shutil.copytree(src, os.path.join(enhanced_zip_dir, ag), dirs_exist_ok=True)

        # Создание архивов
        shutil.make_archive(basic_zip_dir, 'zip', basic_zip_dir)
        shutil.make_archive(enhanced_zip_dir, 'zip', enhanced_zip_dir)
        print(f"Архивы созданы: {basic_zip_dir}.zip, {enhanced_zip_dir}.zip")

        # Автоскачивание в Colab (опционально)
        try:
            from google.colab import files
            files.download(f"{basic_zip_dir}.zip")
            files.download(f"{enhanced_zip_dir}.zip")
        except ImportError:
            print("Не в Colab, архивы не скачиваются автоматически.")

    print("\nОбучение всех RL-агентов завершено")


Training RL agents for ATSP with 12 nodes

  Training PolicyGradient on 12 nodes


PolicyGradient:   0%|          | 0/1500 [00:00<?, ?it/s]/content/rl_agents.py:485: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  'visited_mask': torch.tensor([state['visited_mask']], dtype=torch.float32),
PolicyGradient: 100%|██████████| 1500/1500 [00:22<00:00, 65.41it/s, best=0.00, avg100=0.01]


Метрики сохранены: models_rl_12/PolicyGradient/PolicyGradient_metrics.json

  Training A2C on 12 nodes


A2C: 100%|██████████| 1500/1500 [00:23<00:00, 63.75it/s, best=0.00, avg100=0.01]


Метрики сохранены: models_rl_12/A2C/A2C_metrics.json

  Training PPO on 12 nodes


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Обучение всех RL-агентов завершено
